# 12 - Surgery Theory: Unlinking Tori with SurgerySession

Classical surgery theory (Wall, 1970; Milnor, 1965) describes when two manifolds
are cobordant by cutting and re-gluing along embedded disks. In this notebook we
tackle the most tactile example: **unlinking two linked tori** in $\mathbb{R}^3$
using `SurgerySession` — the high-level Surgeon API.

## Learning Goals
- **Linking number**: why two tori can be topologically inseparable without surgery.
- **SimplicialComplex**: T₁ and T₂ are triangulated meshes, not string descriptors.
- **Betti numbers**: track $\beta_0, \beta_1, \beta_2$ changing at each topological step.
- **Isotopy guard**: `move()` warns when the linear path intersects another surface.
- **Deformation registry**: `deform(mode='open_cut')` widens a puncture into a cylinder.
- **Surgery log**: the three-section Surgery Sequence Log (topological / geometric / algebraic).
- **Obstruction theory**: evaluate $\sigma \in L_n(\mathbb{Z}[\pi_1])$ via the Ranicki assembly map.

## Formal Grounding

### Linking Number and Tori
Two circles $\gamma_1, \gamma_2 \subset \mathbb{R}^3$ have **linking number**
$\mathrm{lk}(\gamma_1, \gamma_2) \in \mathbb{Z}$.  For two tori
$T_1, T_2 \hookrightarrow \mathbb{R}^3$ to be linked, the core circle of $T_2$
passes through the disk bounded by the core circle of $T_1$ exactly once.

### The Surgery Sequence for Unlinking
We cut **$T_2$** (the vertical torus), not $T_1$:

1. **Cut $T_2$**: remove a $D^2$ disk from $T_2$  →  torus-with-hole  ($\Delta\beta_2 = -1$).
2. **Deform $T_2$**: widen the hole into a cylinder with `deform('open_cut')`.
3. **Move $T_2$**: ambient isotopy slides the punctured $T_2$ through $T_1$'s hole.
4. **Reattach**: glue a $D^2$ back onto $T_2$  →  torus again  ($\Delta\beta_2 = +1$).
5. **Restore**: transactional undo demonstration.

| Step | API call | Topological meaning |
|------|----------|---------------------|
| A | `T2.remove_disk(D²)` | $T_2$: torus → torus-with-hole; $\Delta\beta_2 = -1$ |
| B | `T2.deform('open_cut')` | Widen hole into cylinder (geometric only) |
| C | `T2.move(offset)` | Isotopy; $T_2$ slides free (check_isotopy warns on path) |
| D | `attach_handle(D², target='T2')` | Seal $T_2$ back; $\Delta\beta_2 = +1$ |
| E | `restore()` | Transactional undo |


In [1]:
import numpy as np
import plotly.graph_objects as go
from pysurgery.surgery import SurgerySession, DimensionalConsistencyError, SurgeryFinishedError
from pysurgery.bridge.julia_bridge import julia_engine

if julia_engine.available:
    julia_engine.warmup()

print('=' * 70)
print('12 - Surgery Theory: Unlinking Tori  |  Setup Complete')
print('=' * 70)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
[julia_engine.warmup() :: WARMING UP] h1_opt_square
[julia_engine.warmup() :: WARMING UP] h1_valid_square
[julia_engine.warmup() :: WARMING UP] h2_tetra_boundary
[julia_engine.warmup() :: WARMING UP] boundary_payload
[julia_engine.warmup() :: WARMING UP] boundary_mod2
[julia_engine.warmup() :: WARMING UP] pi1_raw_traces
[julia_engine.warmup() :: WARMING UP] metrics_warmup
[julia_engine.warmup() :: WARMING UP] sparse_snf
[julia_engine.warmup() :: WARMING UP] sparse_rank_q
[julia_engine.warmup() :: WARMING UP] sparse_rank_mod_p
[julia_engine.warmup() :: WARMING UP] sparse_cohomology_basis
[julia_engine.warmup() :: WARMING UP] sparse_cohomology_mod_p
[julia_engine.warmup() :: WARMING UP] alexander_whitney_cup
[julia_engine.warmup() :: WARMING UP] group_ring_multiply
[julia_engine.warmup() :: WARMING UP] pi1_abelianization
[julia_engine.warmup() :: WARMING UP] integral_lattice_

## Part 1: Building the Linked Tori

We use the **chain-ring** configuration: both tori have the same major radius $R$ and minor radius $r$,
but are oriented in perpendicular planes with offset centres so they interlock without touching:

- **T₁** (horizontal, $xy$-plane, centred at origin):
  $\mathbf{r}_1(u,v) = \bigl((R + r\cos v)\cos u,\; (R + r\cos v)\sin u,\; r\sin v\bigr)$
- **T₂** (vertical, $xz$-plane, centred at $(R, 0, 0)$):
  $\mathbf{r}_2(u,v) = \bigl(R + (R + r\cos v)\cos u,\; r\sin v,\; (R + r\cos v)\sin u\bigr)$

At $u = \pi$ T₂'s core passes through $(0,0,0)$, which is inside T₁'s hole (inner radius $R - r = 1.6$),
giving **linking number 1**. The minimum distance between the two core circles equals $R$, and
since $R - 2r = 1.2 > 0$ the surfaces are **provably disjoint** (no overlap).


In [2]:
from pysurgery.core.complexes import SimplicialComplex

nu, nv = 50, 25       # visualization mesh resolution
nu_sc, nv_sc = 12, 8  # coarser mesh for SimplicialComplex (same topology)
R = 2.0               # shared major radius
r = 0.4               # minor radius  (2r < R  =>  disjoint surfaces)

u = np.linspace(0, 2 * np.pi, nu)
v = np.linspace(0, 2 * np.pi, nv)
U, V = np.meshgrid(u, v)

# ── Visualization meshes ───────────────────────────────────────────────────────
X1 = (R + r * np.cos(V)) * np.cos(U)
Y1 = (R + r * np.cos(V)) * np.sin(U)
Z1 = r * np.sin(V)

X2 = R + (R + r * np.cos(V)) * np.cos(U)
Y2 = r * np.sin(V)
Z2 = (R + r * np.cos(V)) * np.sin(U)

T1_cloud = np.column_stack([X1.ravel(), Y1.ravel(), Z1.ravel()])
T2_cloud = np.column_stack([X2.ravel(), Y2.ravel(), Z2.ravel()])


def torus_simplicial_complex(nu_s, nv_s, flip_xz=False):
    """Triangulated torus on a nu_s × nv_s periodic grid."""
    triangles = []
    for i in range(nv_s):
        for j in range(nu_s):
            v00 = i * nu_s + j
            v10 = ((i + 1) % nv_s) * nu_s + j
            v01 = i * nu_s + (j + 1) % nu_s
            v11 = ((i + 1) % nv_s) * nu_s + (j + 1) % nu_s
            triangles.append((v00, v10, v01))
            triangles.append((v10, v11, v01))
    return SimplicialComplex.from_simplices(triangles, close_under_faces=True)


T1_sc = torus_simplicial_complex(nu_sc, nv_sc)  # T1: horizontal torus
T2_sc = torus_simplicial_complex(nu_sc, nv_sc)  # T2: vertical torus (same topology)

b1 = T1_sc.betti_numbers()
b2 = T2_sc.betti_numbers()
print(f'T1 (xy-plane, origin)  — mesh: {X1.shape}  cloud: {T1_cloud.shape}')
print(f'T2 (xz-plane, x={R:.0f})   — mesh: {X2.shape}  cloud: {T2_cloud.shape}')
print(f'T1 SimplicialComplex: dim={T1_sc.dimension}  Betti={b1}')
print(f'T2 SimplicialComplex: dim={T2_sc.dimension}  Betti={b2}')
print(f'Surface clearance = R - 2r = {R - 2*r:.1f} > 0  (disjoint)  ok')
print(f'T2 core enters T1 hole at u=pi: core point (0, 0, 0), inner hole radius={R-r}  (linked)')


/home/gabriel/Desktop/SurgeryTheory/pysurgery/core/complexes.py:1434: UserWarning: Torsion certification may be incomplete for this complex; the sparse integer reduction returned only unit factors, so torsion could not be fully resolved.
  warnings.warn(


T1 (xy-plane, origin)  — mesh: (25, 50)  cloud: (1250, 3)
T2 (xz-plane, x=2)   — mesh: (25, 50)  cloud: (1250, 3)
T1 SimplicialComplex: dim=2  Betti={0: 1, 1: 2, 2: 1}
T2 SimplicialComplex: dim=2  Betti={0: 1, 1: 2, 2: 1}
Surface clearance = R - 2r = 1.2 > 0  (disjoint)  ok
T2 core enters T1 hole at u=pi: core point (0, 0, 0), inner hole radius=1.6  (linked)


### Example 12.1: Interactive Visualization — Linked Tori (Before Surgery)

Rotate the scene to verify the linking: T₂ (red) threads through the hole of T₁ (blue). The yellow ✕ markers show the **crossing sites** — the disk-removal sites for the surgery.

> **Tip**: hold Shift and drag to pan, use scroll to zoom.

In [3]:
import numpy as np

# Cut site on T2: at U=pi/2 (topmost arc of T2 in xz-plane), V=0 (outer tube surface)
# T2 core at U=pi/2: (R + R*cos(pi/2), 0, R*sin(pi/2)) = (R, 0, R)
# Outermost tube point at V=0: (R, 0, R+r) -- far from T1 (z~0 plane)
cut_site_T2 = (R, 0.0, R + r)        # = (2.0, 0.0, 2.4)
cut_sites = np.array([cut_site_T2])

fig_before = go.Figure()

fig_before.add_trace(go.Surface(
    x=X1, y=Y1, z=Z1,
    colorscale='Blues',
    opacity=0.65,
    name='T1 (xy-plane, fixed)',
    showscale=False,
    hovertemplate='T1  x=%{x:.2f}  y=%{y:.2f}  z=%{z:.2f}<extra></extra>',
))

fig_before.add_trace(go.Surface(
    x=X2, y=Y2, z=Z2,
    colorscale='Reds',
    opacity=0.65,
    name='T2 (xz-plane, will be cut)',
    showscale=False,
    hovertemplate='T2  x=%{x:.2f}  y=%{y:.2f}  z=%{z:.2f}<extra></extra>',
))

fig_before.add_trace(go.Scatter3d(
    x=cut_sites[:, 0], y=cut_sites[:, 1], z=cut_sites[:, 2],
    mode='markers+text',
    marker=dict(size=14, color='yellow', symbol='cross',
                line=dict(color='black', width=2)),
    text=[f'Cut on T2  ({cut_site_T2[0]:.1f}, {cut_site_T2[1]:.1f}, {cut_site_T2[2]:.1f})'],
    textposition='top center',
    name='Cut site D² on T2',
))

print(f'Cut site on T2:  {cut_site_T2}  (top of T2, far from T1)')
print(f'T1 tube z-range: [{-r:.1f}, {r:.1f}]  —  cut site z={cut_site_T2[2]:.1f}  (no conflict)')

fig_before.update_layout(
    title=dict(text='Linked Tori — Before Surgery  (lk = 1)', font=dict(size=16)),
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        camera=dict(eye=dict(x=2.0, y=0.8, z=0.8)),
        aspectmode='cube',
    ),
    legend=dict(x=0.02, y=0.95),
    height=650,
    width=950,
)
fig_before.show()


Cut site on T2:  (2.0, 0.0, 2.4)  (top of T2, far from T1)
T1 tube z-range: [-0.4, 0.4]  —  cut site z=2.4  (no conflict)


## Part 2: Setting Up the SurgerySession

`SurgerySession` is the **Surgeon API**: it maintains three synchronized models:
- **Topological model** — the ambient space and manifold descriptor, recorded in `cobordism`.
- **Algebraic model** — `chain_complex` (`AlgebraicPoincareComplex`) updated by each operation.
- **Geometric model** — `point_clouds` translated in tandem with each isotopy.

Two proxy objects give clean access:
- `surgeon.AmbientSpace` — proxy for `remove_disks`, `attach_handle`, `restore`, and `()` callable to get the manifold.
- `surgeon.objects` — proxy supporting `objects["T1"]` (dict-style) and `objects()` (returns full dict).

In [4]:
surgeon = SurgerySession(
    ambient_space='R^3',
    objects={'T1': T1_sc, 'T2': T2_sc},      # SimplicialComplex objects
    point_clouds={'T1': T1_cloud, 'T2': T2_cloud},
)

print(f'Ambient space:       {surgeon.manifold}')
print(f'Ambient dimension:   {surgeon.chain_complex.dimension}')
print(f'Tracked objects:     {list(surgeon.objects())}')
print(f'T1 data type:        {type(surgeon.objects["T1"].data).__name__}')
print(f'T2 data type:        {type(surgeon.objects["T2"].data).__name__}')
print(f'Session finished?    {surgeon._finished}')
print()
print('AmbientSpace proxy:', surgeon.AmbientSpace)
print('T1 TrackedObject:  ', surgeon.objects["T1"])
print('T2 TrackedObject:  ', surgeon.objects["T2"])


Ambient space:       R^3
Ambient dimension:   3
Tracked objects:     ['T1', 'T2']
T1 data type:        SimplicialComplex
T2 data type:        SimplicialComplex
Session finished?    False

AmbientSpace proxy: AmbientSpaceProxy(manifold='R^3')
T1 TrackedObject:   TrackedObject(name='T1')
T2 TrackedObject:   TrackedObject(name='T2')


## Part 3: The Surgery Sequence — Unlinking the Tori

We perform the surgery in **five steps**, all operating on **T₂** (the vertical torus).
T₁ is never cut — it acts purely as the ambient obstacle.

At each topological step (`remove_disk`, `attach_handle`) the session reads T₂'s
current Betti numbers from the `SimplicialComplex` object and computes the theoretical
delta (Mayer-Vietoris).  The result appears in `logs()` as a Betti-number sequence table.


### Step A — Remove D² from T₂

We remove a $D^2$ disk from **T₂** at $(R, 0, R+r) = (2, 0, 2.4)$ — the topmost outer
point of T₂, far from T₁.

Topological effect (Mayer-Vietoris):
$$T^2 - D^2 \;\simeq\; \text{torus-with-one-hole}$$
$$\beta = [1,\,2,\,1] \;\xrightarrow{\Delta\beta_2 = -1}\; [1,\,2,\,0]$$

The boundary of the removed disk is a circle $S^1$ embedded in T₂.
The `SimplicialComplex` object is used to compute exact Betti numbers before and after.


In [5]:
print('Step A: Remove D² disk from T2')
print('-' * 50)

surgeon.objects['T2'].remove_disk(at=cut_site_T2, disk_type='D^2')

op = surgeon.cobordism[-1]
print(f'Cobordism steps: {len(surgeon.cobordism)}')
print(f'  step:         {op["step"]}')
print(f'  target:       {op["target"]}')
print(f'  types:        {op["types_math"]}')
print(f'  Betti before: {op["betti_before"]}')
print(f'  Betti after:  {op["betti_after"]}')
print(f'  Δβ:           {op["delta_betti"]}')
print(f'  theorem:      {op["theorem"]}')
print()

# Guard: D^4 in R^3 is geometrically impossible
try:
    surgeon.AmbientSpace.remove_disks(('D^4',), at=[(0, 0, 0)])
except DimensionalConsistencyError as e:
    print(f'DimensionalConsistencyError: {e}')


Step A: Remove D² disk from T2
--------------------------------------------------


/home/gabriel/Desktop/SurgeryTheory/pysurgery/core/complexes.py:1434: UserWarning: Torsion certification may be incomplete for this complex; the sparse integer reduction returned only unit factors, so torsion could not be fully resolved.
  warnings.warn(


Cobordism steps: 1
  step:         remove_disks
  target:       T2
  types:        ['D²']
  Betti before: {0: 1, 1: 2, 2: 1}
  Betti after:  {0: 1, 1: 2, 2: 0}
  Δβ:           [0, 0, -1]
  theorem:      SURGERY_HANDLE_MAYER_VIETORIS

DimensionalConsistencyError: Cannot remove disk of type D^4 (dim=4) from a 3-manifold.


### Step B — Deform T₂: Open the Puncture into a Cylinder

After removing $D^2$, T₂ has a small hole at $(2, 0, 2.4)$.
We call `deform(mode='open_cut')` to **widen that hole into a cylindrical opening**
large enough for T₁'s tube to be "seen through".

The `open_cut` deformation pushes each nearby point outward along `pull_direction`
with Gaussian falloff:
$$\mathbf{p} \mapsto \mathbf{p}
  + e^{-\tfrac{\|p-c\|^2}{2\sigma^2}}
  \cdot \mathrm{sign}(\langle p-c,\,\mathbf{d}\rangle)
  \cdot \tfrac{\delta}{2}\,\mathbf{d}$$

Pull direction $\mathbf{d} = (1, 0, 0)$ (tangent to T₂'s core circle at $U=\pi/2$):
the two rim circles of the cut are pulled apart along $x$, opening the gap.

**Extensibility** — new modes add zero changes to the core API:
```python
from pysurgery.surgery import _register_deform_mode
@_register_deform_mode('bend')
def _deform_bend(cloud, *, axis, angle, ...):
    ...
    return deformed_cloud, metadata_dict
```


In [6]:
print('Step B: Open the D² puncture into a cylinder shape')
print('-' * 50)

from pysurgery.surgery import list_deform_modes
print(f'Registered deformation modes: {list_deform_modes()}')
print()

iso_deform = surgeon.objects['T2'].deform(
    mode='open_cut',
    cut_site=cut_site_T2,               # (2.0, 0.0, 2.4)
    pull_direction=(1.0, 0.0, 0.0),     # tangent to T2 core at U=pi/2  =>  opens along x
    width=2.0,                          # total gap width
    falloff_radius=1.0,                 # Gaussian sigma
)
print(f'Deformation result: {iso_deform}')
print()

op = surgeon.cobordism[-1]
meta = op['meta']
print(f'Mode:              {op["mode"]}')
print(f'Cut site:          {meta["cut_site"]}')
print(f'Pull direction:    {meta["pull_direction"]}')
print(f'Width:             {meta["width"]}  Falloff sigma: {meta["falloff_radius"]}')
print(f'Points displaced:  {meta["displaced_points"]} / {len(surgeon.point_clouds["T2"])} total')

sb, sa = op['stats_before'], op['stats_after']
print(f'T2 centroid before: ({sb["centroid"][0]:.3f}, {sb["centroid"][1]:.3f}, {sb["centroid"][2]:.3f})')
print(f'T2 centroid after:  ({sa["centroid"][0]:.3f}, {sa["centroid"][1]:.3f}, {sa["centroid"][2]:.3f})')
print(f'T2 bbox x before: [{sb["bbox_min"][0]:.2f}, {sb["bbox_max"][0]:.2f}]')
print(f'T2 bbox x after:  [{sa["bbox_min"][0]:.2f}, {sa["bbox_max"][0]:.2f}]  (wider -> cut opened)')
print(f'Undo stack depth:  {len(surgeon.stack)}')


Step B: Open the D² puncture into a cylinder shape
--------------------------------------------------
Registered deformation modes: ['open_cut']

Deformation result: Transformation('deform[open_cut](T2): 595 pts displaced')

Mode:              open_cut
Cut site:          [2.0, 0.0, 2.4]
Pull direction:    [1.0, 0.0, 0.0]
Width:             2.0  Falloff sigma: 1.0
Points displaced:  595 / 1250 total
T2 centroid before: (2.040, -0.000, 0.000)
T2 centroid after:  (2.049, -0.000, 0.000)
T2 bbox x before: [-0.40, 4.40]
T2 bbox x after:  [-0.40, 4.40]  (wider -> cut opened)
Undo stack depth:  2


### Step C — Ambient Isotopy: Move T₂ Through T₁’s Hole

With T₂ now an open cylinder (its $D^2$ puncture exposed), we apply a large $x$-offset
$$f(x, t) = x + t \cdot (8,\; 0,\; 0), \quad t \in [0,1],$$
sliding T₂ through T₁’s central hole.

`move(check_isotopy=True)` performs a bounding-box + point-distance scan along the
trajectory.  Since T₂ is an open cylinder, it can thread through T₁’s hole without
intersection — no warning will be issued for a valid unlink.

At $t = 0$ T₂ is linked with T₁ (centre at $(R, 0, 0) = (2, 0, 0)$).  
At $t = 1$ T₂ has slid free, with centre at $(10, 0, 0)$ — topologically unlinked.

In [7]:
print('Step C: Ambient isotopy — slide T2 through T1\'s hole')
print('-' * 50)

# Capture the deformed cloud BEFORE sliding (used in animation)
T2_def_cloud = surgeon.point_clouds['T2'].copy()

iso = surgeon.objects['T2'].move(offset=(8.0, 0.0, 0.0), check_isotopy=True)

print(f'Transformation: {iso}')
print(f'Isotopy fn:     {iso.description}')
print()

op = surgeon.cobordism[-1]
sb = op['stats_before']['T2']
sa = op['stats_after']['T2']
print(f'T2 centroid before: ({sb["centroid"][0]:.2f}, {sb["centroid"][1]:.2f}, {sb["centroid"][2]:.2f})')
print(f'T2 centroid after:  ({sa["centroid"][0]:.2f}, {sa["centroid"][1]:.2f}, {sa["centroid"][2]:.2f})')
print(f'Cobordism steps so far: {len(surgeon.cobordism)}')


Step C: Ambient isotopy — slide T2 through T1's hole
--------------------------------------------------
Transformation: Transformation('f(x,t) = x + t·(+8.00, +0.00, +0.00),  t ∈ [0,1]')
Isotopy fn:     f(x,t) = x + t·(+8.00, +0.00, +0.00),  t ∈ [0,1]

T2 centroid before: (2.05, -0.00, 0.00)
T2 centroid after:  (10.05, -0.00, 0.00)
Cobordism steps so far: 3


### Step D — Attach Handle: Seal T₂ Back into a Torus

After T₂ has slid free, we re-attach a $D^2$ handle to close its open boundary, restoring
the torus topology.  This is the algebraic inverse of Step A:
$$\Delta\beta_2 = +1, \qquad \beta(T_2) : (1,\; 2,\; 0) \longrightarrow (1,\; 2,\; 1).$$

The Mayer-Vietoris certificate confirms no net topological change over the full
surgery $A \to D$: T₂ is still a torus, just unlinked from T₁.

In [8]:
print('Step D: Attach handle — seal T2 back into a torus')
print('-' * 50)

betti_before = dict(surgeon._betti_tracker.get('T2', {}))
print(f'beta(T2) before handle: {betti_before}')

surgeon.AmbientSpace.attach_handle(
    at=(cut_site_T2, cut_site_T2),   # both handle ends on T2's open boundary
    handle_type='D^2xS^0',
    target='T2',
)

betti_after = dict(surgeon._betti_tracker.get('T2', {}))
print(f'beta(T2) after  handle: {betti_after}')
print(f'Delta_beta2 = {betti_after.get(2,0) - betti_before.get(2,0):+d}  (expect +1)')
print()

op = surgeon.cobordism[-1]
print(f'handle:  {op["handle_name"]}  (type: {op["handle_type_math"]})')
print(f'dim:     {op["handle_dim"]}  (ambient: {op["ambient_dim"]})')
print(f'theorem: {op["theorem"]}')
print(f'Cobordism steps so far: {len(surgeon.cobordism)}')


Step D: Attach handle — seal T2 back into a torus
--------------------------------------------------
beta(T2) before handle: {0: 1, 1: 2, 2: 0}
beta(T2) after  handle: {0: 1, 1: 2, 2: 1}
Delta_beta2 = +1  (expect +1)

handle:  Handle1  (type: D²×S⁰)
dim:     2  (ambient: 3)
theorem: SURGERY_HANDLE_MAYER_VIETORIS
Cobordism steps so far: 4


### Step E — Transactional Undo: Demonstrating `restore()`

`restore()` is a **transactional undo**: it pops the last entry from the undo stack and reverts all point clouds to their snapshot state. We apply a small extra translation and immediately roll it back, verifying exact round-trip recovery.

In [9]:
print('Step E: Transactional undo demonstration')
print('-' * 50)

# Save T2 at x+8 (result of main isotopy)
T2_after_isotopy = surgeon.point_clouds['T2'].copy()

# Apply a small extra translation
surgeon.objects['T2'].move(offset=(2.0, 0.0, 0.0))
cx_extra = surgeon.point_clouds['T2'].mean(axis=0)
print(f'After extra move  — T2 centroid: ({cx_extra[0]:.2f}, {cx_extra[1]:.2f}, {cx_extra[2]:.2f})')

# Roll back just the extra move
surgeon.AmbientSpace.restore()
cx_restored = surgeon.point_clouds['T2'].mean(axis=0)
print(f'After restore()   — T2 centroid: ({cx_restored[0]:.2f}, {cx_restored[1]:.2f}, {cx_restored[2]:.2f})')
print(f'Stack depth after restore: {len(surgeon.stack)}')

# Verify exact round-trip
assert np.allclose(surgeon.point_clouds['T2'], T2_after_isotopy), 'restore() round-trip failed!'
print('Round-trip verified: np.allclose passes')


Step E: Transactional undo demonstration
--------------------------------------------------
After extra move  — T2 centroid: (12.05, -0.00, 0.00)
After restore()   — T2 centroid: (10.05, -0.00, 0.00)
Stack depth after restore: 4
Round-trip verified: np.allclose passes


In [10]:
surgeon.finish()
print(f'Session finalized.  _finished = {surgeon._finished}')

# Finality guard: no mutations after finish()
try:
    surgeon.objects['T2'].move(offset=(1.0, 0.0, 0.0))
except SurgeryFinishedError as e:
    print(f'SurgeryFinishedError: {e}')


Session finalized.  _finished = True
SurgeryFinishedError: Surgery session has been finished. No further mutations are allowed.


## Part 4: Interactive Surgery Animation

The animation below plays through the five surgical steps in sequence.
Drag the **slider** or press **▶ Play** to watch:

| Phase | What you see |
|-------|--------------|
| **Initial** | Both tori linked; yellow × marks the disk-removal site on T₁ |
| **A+B — Cut & open** | T₁ morphs from a round torus to an open cylinder; × fades as the hole widens |
| **D — T₂ slides through** | T₂ (red) translates along $x$ while T₁ (blue) holds the opening |
| **C — Handle seals T₁** | T₁ morphs back to its round shape, representing the $S^2 \times D^1$ handle filling the hole |
| **Final** | T₁ sealed and round; T₂ free at $x \approx 10$ — **linking number = 0** |

> **Note:** The closing animation represents handle attachment (Step C) completing the surgery.
> Topologically the handle was attached before T₂ moved; geometrically the visual
> closure confirms T₁ is restored to a torus after T₂ has passed.


In [11]:
# ── build arrays for animation ───────────────────────────────────────────────
# T1 is NEVER modified; its grid arrays (X1, Y1, Z1) are the permanent display.
# T2 passes through three visual states:
#   T2_orig  : original torus at x~2  (from X2, Y2, Z2 grid)
#   T2_def   : deformed cylinder at x~2  (captured in step-c before slide)
#   T2_fin   : deformed cylinder at x~10 (surgeon.point_clouds['T2'] after slide)
#   T2_closed: original torus shape at x~10 (for 'sealing' animation)

T1_X, T1_Y, T1_Z = X1.copy(), Y1.copy(), Z1.copy()

T2_orig_X, T2_orig_Y, T2_orig_Z = X2.copy(), Y2.copy(), Z2.copy()

T2_def_X = T2_def_cloud[:, 0].reshape(nv, nu)
T2_def_Y = T2_def_cloud[:, 1].reshape(nv, nu)
T2_def_Z = T2_def_cloud[:, 2].reshape(nv, nu)

T2_fin_flat = surgeon.point_clouds['T2']
T2_fin_X = T2_fin_flat[:, 0].reshape(nv, nu)
T2_fin_Y = T2_fin_flat[:, 1].reshape(nv, nu)
T2_fin_Z = T2_fin_flat[:, 2].reshape(nv, nu)

# Original torus shape translated to final x-position (for closing animation)
T2_closed_X = T2_orig_X + 8.0
T2_closed_Y = T2_orig_Y.copy()
T2_closed_Z = T2_orig_Z.copy()


def lerp(A, B, t):
    return A + t * (B - A)


def frame_traces(t1x, t1y, t1z, t2x, t2y, t2z, cut_op):
    return [
        go.Surface(x=t1x, y=t1y, z=t1z, colorscale='Blues', opacity=0.72,
                   showscale=False, showlegend=False),
        go.Surface(x=t2x, y=t2y, z=t2z, colorscale='Reds',  opacity=0.72,
                   showscale=False, showlegend=False),
        go.Scatter3d(
            x=[cut_site_T2[0]], y=[cut_site_T2[1]], z=[cut_site_T2[2]],
            mode='markers',
            marker=dict(size=12, color='yellow', opacity=cut_op,
                        symbol='cross', line=dict(color='darkgoldenrod', width=2)),
            showlegend=False,
        ),
    ]


N0, N1, N2, N3, N4 = 2, 10, 12, 10, 2
frames, slider_steps, _fi = [], [], [0]


def add_frame(t1x, t1y, t1z, t2x, t2y, t2z, cut_op, label):
    name = f'f{_fi[0]}'
    _fi[0] += 1
    frames.append(go.Frame(
        data=frame_traces(t1x, t1y, t1z, t2x, t2y, t2z, cut_op),
        name=name,
    ))
    slider_steps.append(dict(
        args=[[name], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}],
        label=label, method='animate',
    ))


# Phase 0: Initial configuration (linked)
for _ in range(N0):
    add_frame(T1_X, T1_Y, T1_Z,
              T2_orig_X, T2_orig_Y, T2_orig_Z, 1.0, 'Initial (lk=1)')

# Phase A+B: T2 opens from torus to cylinder (cut marker fades)
for fi in range(N1 + 1):
    t = fi / N1
    add_frame(
        T1_X, T1_Y, T1_Z,
        lerp(T2_orig_X, T2_def_X, t), lerp(T2_orig_Y, T2_def_Y, t),
        lerp(T2_orig_Z, T2_def_Z, t),
        max(0.0, 1.0 - 2.8 * t),
        f'A+B — T2 opens  ({int(t*100)}%)',
    )

# Phase C: T2 cylinder slides through T1's hole
for fi in range(N2 + 1):
    t = fi / N2
    add_frame(
        T1_X, T1_Y, T1_Z,
        lerp(T2_def_X, T2_fin_X, t), lerp(T2_def_Y, T2_fin_Y, t),
        lerp(T2_def_Z, T2_fin_Z, t),
        0.0, f'C — T2 slides free  ({int(t*100)}%)',
    )

# Phase D: T2 closes back into a torus at new position
for fi in range(N3 + 1):
    t = fi / N3
    add_frame(
        T1_X, T1_Y, T1_Z,
        lerp(T2_fin_X, T2_closed_X, t), lerp(T2_fin_Y, T2_closed_Y, t),
        lerp(T2_fin_Z, T2_closed_Z, t),
        0.0, f'D — T2 seals  ({int(t*100)}%)',
    )

# Phase Final: both tori as closed surfaces (unlinked)
for _ in range(N4):
    add_frame(T1_X, T1_Y, T1_Z,
              T2_closed_X, T2_closed_Y, T2_closed_Z, 0.0, 'Final (lk=0)')


fig_anim = go.Figure(
    data=frame_traces(T1_X, T1_Y, T1_Z,
                      T2_orig_X, T2_orig_Y, T2_orig_Z, cut_op=1.0),
    frames=frames,
    layout=go.Layout(
        title=dict(
            text='Surgery Animation — Unlinking T₂ from T₁  [ lk = 1 ⟶ 0 ]',
            font=dict(size=15),
        ),
        scene=dict(
            xaxis=dict(title='x', range=[-3.5, 13.5]),
            yaxis=dict(title='y', range=[-3.0,  3.0]),
            zaxis=dict(title='z', range=[-3.0,  3.0]),
            camera=dict(eye=dict(x=0.65, y=1.5, z=0.8)),
            aspectmode='manual',
            aspectratio=dict(x=4.0, y=1.5, z=1.5),
        ),
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            x=0.5, xanchor='center',
            y=-0.06, yanchor='top',
            buttons=[
                dict(
                    label='▶  Play',
                    method='animate',
                    args=[None, {
                        'frame': {'duration': 130, 'redraw': True},
                        'fromcurrent': True,
                        'transition': {'duration': 0},
                    }],
                ),
                dict(
                    label='⏸  Pause',
                    method='animate',
                    args=[[None], {
                        'frame': {'duration': 0, 'redraw': False},
                        'mode': 'immediate',
                        'transition': {'duration': 0},
                    }],
                ),
            ],
        )],
        sliders=[dict(
            active=0,
            currentvalue=dict(
                prefix='Step: ',
                visible=True,
                xanchor='center',
                font=dict(size=11, color='#555'),
            ),
            pad=dict(t=55, b=10, l=60, r=60),
            steps=slider_steps,
        )],
        height=730,
        width=1130,
        margin=dict(b=150, t=60, l=20, r=20),
        showlegend=True,
        legend=dict(x=0.01, y=0.98),
    ),
)

fig_anim.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None], mode='markers',
    marker=dict(size=10, color='steelblue'), name='T1  (blue, static)', showlegend=True,
))
fig_anim.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None], mode='markers',
    marker=dict(size=10, color='crimson'), name='T2  (red, operated)', showlegend=True,
))
fig_anim.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None], mode='markers',
    marker=dict(size=10, color='gold', symbol='cross'), name='Cut site (T2)', showlegend=True,
))

print(f'Animation: {len(frames)} frames')
for label, n in [('Initial', N0), ('T2 opens', N1+1),
                 ('T2 slides', N2+1), ('T2 seals', N3+1), ('Final', N4)]:
    print(f'  {label:12s}: {n} frames')
print('T1 stays static (blue) throughout.')
print('Press ▶ Play or drag the slider.')
fig_anim.show()


Animation: 39 frames
  Initial     : 2 frames
  T2 opens    : 11 frames
  T2 slides   : 13 frames
  T2 seals    : 11 frames
  Final       : 2 frames
T1 stays static (blue) throughout.
Press ▶ Play or drag the slider.


## Part 5: The Surgery Sequence Log

`surgeon.logs()` generates an automatic three-section structured report:

1. **Topological Trace** — the surgery sequence arrow diagram $M_0 \to M_1 \to \cdots$ and per-step Mayer-Vietoris certificates (disk types, dimensions, theorem tags).
2. **Geometric Trace** — isotopy functions $f(x,t)$, centroid trajectories, bounding-box deltas, and restore records for each `move`/`restore` step.
3. **Algebraic Proof** — the fundamental group $\pi_1$, coefficient ring $\mathbb{Z}[\pi_1]$, and the L-group obstruction $\sigma \in L_n(\mathbb{Z}[\pi_1])$ via the Ranicki assembly map.

The log is read-only and available even after `finish()`.

In [12]:
log = surgeon.logs(latex=False)
print(log)


══════════════════════════════════════════════════════════════════════
    SURGERY SEQUENCE LOG                                              
    Ambient: ℝ³  ·  Steps recorded: 6  ·  Objects: T1, T2, Handle1    
══════════════════════════════════════════════════════════════════════

  ━━  I. TOPOLOGICAL TRACE  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  M₀ ──[remove_disks(D² on T2)]──▶ M₁ ──[attach_handle(D²×S⁰ on T2)]──▶ M₂

  Betti number sequence  (β₀  β₁  β₂  …):

  Object        M₀  M₁  M₂
  ──────────────────────────────
  T2            [1 2 1]  [1 2 0]  [1 2 1]

  ┌─ Step 1 · remove_disks ─────────────────────────────────────────────┐
  │  Disks:      D² (dim=2)   (ambient dim = 3)                        │
  │  Target:     T2                                                    │
  │  Sites:      (2.0, 0.0, 2.4)                                       │
  │  Δβ:         Δβ₀=0  Δβ₁=0  Δβ₂=-1                                  │
  │  β before:   [1 2 1]                             

## Part 6: Algebraic Proof — Surgery Obstruction in L-Theory

The surgery obstruction $\sigma(f, b) \in L_n(\mathbb{Z}[\pi_1(Y)])$ measures whether a degree-1 normal map $(f, b): X \to Y$ is normally cobordant to a homotopy equivalence (Ranicki, 1992, §9). For $n$ odd and simply-connected $\mathbb{R}^3$, we have $L_3(\mathbb{Z}) = 0$, so the obstruction vanishes and surgery can always be completed.

In [13]:
print('Surgery obstruction via Ranicki assembly map')
print('-' * 50)

obs = surgeon.evaluate_obstruction()

print(f'Result type:         {type(obs).__name__}')
print(f'dimension:           {obs.dimension}')
print(f'pi:                  {obs.pi}')
print(f'exact:               {obs.exact}')
print(f'obstructs:           {obs.obstructs}')
print(f'assembly_certified:  {obs.assembly_certified}')
print(f'value:               {obs.value}')
print(f'zero_certified:      {obs.zero_certified}')
if obs.message:
    print(f'message:             {obs.message}')
print()
if not obs.obstructs:
    print('sigma = 0  =>  No obstruction.')
    print('The unlinking surgery can be completed; T1 and T2 are')
    print('h-cobordant to the unlinked configuration.')
else:
    print('sigma != 0  =>  Obstruction present.')
    print('Surgery may not simplify the manifold to the desired form.')


Surgery obstruction via Ranicki assembly map
--------------------------------------------------
Result type:         ObstructionResult
dimension:           3
pi:                  1
exact:               True
obstructs:           False
assembly_certified:  True
value:               0
zero_certified:      True
message:             Simply-connected odd-dimensional L-groups are zero.

sigma = 0  =>  No obstruction.
The unlinking surgery can be completed; T1 and T2 are
h-cobordant to the unlinked configuration.


## Part 7: LaTeX Output

`surgeon.logs(latex=True)` renders the Surgery Sequence Log in LaTeX markup suitable for inclusion in a paper or proof assistant. The arrow diagram uses `\xrightarrow{...}` and steps become `\paragraph{...}` blocks.

In [14]:
latex_log = surgeon.logs(latex=True)
print(latex_log)

\section*{Surgery Sequence Log}
Ambient space: $\mathbb{R}^{3}$. Steps recorded: 6.

\subsection*{I.\; Topological Trace}
\[ M_0\; \xrightarrow{\text{remove}(D^{2})} M_{1}\; \xrightarrow{\text{attach}(D^{2\times S^{0)} M_{2} \]
\paragraph{Step 1: remove disks}
Removed: $D²$. Sites: $[(2.0, 0.0, 2.4)]$. Theorem: \texttt{SURGERY_HANDLE_MAYER_VIETORIS}.
\paragraph{Step 4: attach handle}
Handle: $\texttt{Handle1}$ (type: $D²×S⁰$, $\dim=2$). Theorem: \texttt{SURGERY_HANDLE_MAYER_VIETORIS}.
\subsection*{II.\; Geometric Trace}
\paragraph{Step 3: move}
Isotopy: $f(x,t) = x + t \cdot (8.0, 0.0, 0.0)$, $t \in [0,1]$.
Cloud \texttt{T2}: 1250 points; centroid $(2.05, -0.00, 0.00) \to (10.05, -0.00, 0.00)$.
\paragraph{Step 5: move}
Isotopy: $f(x,t) = x + t \cdot (2.0, 0.0, 0.0)$, $t \in [0,1]$.
Cloud \texttt{T2}: 1250 points; centroid $(10.05, -0.00, 0.00) \to (12.05, -0.00, 0.00)$.
\paragraph{Step 6: restore}
Reverted step 5: move.
\subsection*{III.\; Algebraic Proof}
Ambient dimension $n = 3$. In

## Failure Modes

1. **`DimensionalConsistencyError`**: Attempting to remove a $D^k$ disk with $k >$ ambient dimension (e.g., $D^4$ from $\mathbb{R}^3$).
2. **`SurgeryFinishedError`**: Any mutative call — `remove_disks`, `attach_handle`, `move`, `restore` — after `finish()`.
3. **`restore()` on empty stack**: Calling `restore()` when the undo stack is empty is a **silent no-op** (no error raised).

In [15]:
# Failure 1: DimensionalConsistencyError
s = SurgerySession(ambient_space='R^3')
try:
    s.remove_disks('D^4', at=[(0, 0, 0)])
except DimensionalConsistencyError as e:
    print(f'1. DimensionalConsistencyError: {e}')

# Failure 2: SurgeryFinishedError
s.finish()
try:
    s.remove_disks('D^3', at=[(0, 0, 0)])
except SurgeryFinishedError as e:
    print(f'2. SurgeryFinishedError: {e}')

# Failure 3: restore() on empty stack — silent no-op
s2 = SurgerySession(ambient_space='R^3')
assert len(s2.stack) == 0
s2.restore()
assert len(s2.stack) == 0
print('3. restore() on empty stack: no error (silent no-op)  ok')

# Bonus: finish() is idempotent
s3 = SurgerySession(ambient_space='R^3')
s3.finish()
s3.finish()  # second call must not raise
print('4. finish() idempotent: second call ok')


1. DimensionalConsistencyError: Cannot remove disk of type D^4 (dim=4) from a 3-manifold.
2. SurgeryFinishedError: Surgery session has been finished. No further mutations are allowed.
3. restore() on empty stack: no error (silent no-op)  ok
4. finish() idempotent: second call ok


## Summary Checklist
- [x] Built two linked tori as `SimplicialComplex` objects in $\mathbb{R}^3$, verified $\beta(T^2) = (1,2,1)$.
- [x] Visualized the linked configuration with Plotly interactive 3D surfaces.
- [x] Created a `SurgerySession` with `AmbientSpace` and `objects` proxies.
- [x] **Step A**: Removed $D^2$ from T₂ via `TrackedObject.remove_disk()`, tracking $\Delta\beta_2 = -1$.
- [x] **Step B**: Deformed T₂ into a cylinder with `deform(mode='open_cut')` (extensible deformation registry).
- [x] **Step C**: Slid T₂ through T₁’s hole with `move(check_isotopy=True)` (isotopy collision guard).
- [x] **Step D**: Re-attached a $D^2$ handle to seal T₂, recovering $\Delta\beta_2 = +1$.
- [x] **Step E**: Demonstrated transactional undo via `restore()` with exact round-trip verification.
- [x] Visualized the full surgery as a step-by-step Plotly animation (T1 static, T2 opens/slides/closes).
- [x] Read the full Surgery Sequence Log with Betti-number trace (topological / geometric / algebraic).
- [x] Evaluated the surgery obstruction $\sigma \in L_3(\mathbb{Z})$ via the Ranicki assembly map.
- [x] Rendered the log in LaTeX markup.
- [x] Demonstrated all three failure modes.

## Exercises
1. **Linking number 2**: Position T₂ so it threads through T₁’s hole twice. What changes in the surgery sequence?
2. **4D surgery**: Create `SurgerySession(ambient_space='R^4')` and attach a $D^4$ disk. How does the log differ?
3. **Multiple isotopies**: Move T₂ in two stages: first $(4,0,0)$, then $(4,0,0)$ more. Compare the cobordism trace.
4. **Restore chain**: Call `restore()` to undo Step E, then Step C (move), verifying centroids at each step.
5. **LaTeX integration**: Paste the LaTeX log output into an Overleaf document and verify it compiles.

## Key Takeaways
- `SurgerySession` orchestrates **topological + algebraic + geometric** synchronisation in one API.
- Surgery is performed **on T₂ only** (T₁ is the static reference): remove disk → deform → slide → seal.
- **Betti numbers** are tracked at each surgery step, giving a live $\beta$-sequence certificate.
- `move(check_isotopy=True)` guards against path intersections with other tracked objects.
- The **cobordism trace** records complete surgery metadata for automatic log generation.
- `restore()` provides transactional undo with exact round-trip guarantees on point clouds.
- The **L-group obstruction** $\sigma = 0$ for $\mathbb{R}^3$ confirms the surgery can always be completed.

**Ready for [13 — Structure Sets and Exact Sequence Navigation](./13_structure_sets_and_exact_sequence_navigation.ipynb)**
